# Runtime Containment for Multi-Agent Workflows

When you run multiple agents in a loop -- researching, planning, executing -- nothing stops them from blowing through your API budget or looping forever. OpenAI's [guardrails](https://platform.openai.com/docs/guides/agent-builder-safety) handle content safety, but not execution limits: how many calls, how much spend, when to stop.

This notebook shows how to add runtime containment to a multi-agent workflow built with the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python). We use [veronica-core](https://github.com/amabito/veronica-core) as the containment layer -- you can swap in any library that does budget tracking and circuit breaking.

We will:
1. Run a multi-agent research workflow **without** containment and watch it overshoot
2. Run the **same** workflow **with** containment and see it halt cleanly
3. Compare cost, call count, and execution time side by side

## Setup

1. Install dependencies

In [ ]:
%pip install openai-agents veronica-core nest_asyncio -q

In [ ]:
import os
import asyncio
import time
import nest_asyncio

from agents import Agent, Runner

nest_asyncio.apply()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"

2. Define the agents

In [ ]:
research_agent = Agent(
    name="Researcher",
    instructions=(
        "You are a research assistant. When given a topic, "
        "search for key facts and return a structured summary. "
        "If you need more detail, ask follow-up questions."
    ),
)

writer_agent = Agent(
    name="Writer",
    instructions=(
        "You are a technical writer. Take research notes and "
        "produce a concise, well-structured report. "
        "If the notes are incomplete, request more research."
    ),
)

reviewer_agent = Agent(
    name="Reviewer",
    instructions=(
        "You are a senior editor. Review the report for accuracy "
        "and completeness. If issues are found, send it back to "
        "the writer with specific feedback. Be thorough -- check "
        "every claim."
    ),
)

## Part 1: Without Containment

We run a research-write-review loop. Each iteration calls all three agents. Without limits, the loop runs until `max_iterations` -- burning through API calls even when the output is already good enough.

3. Run the uncontained loop

In [ ]:
async def run_uncontained(topic, max_iterations=6):
    stats = {"calls": 0, "start": time.time()}
    notes = topic
    report = ""

    for i in range(max_iterations):
        result = await Runner.run(research_agent, notes)
        notes = result.final_output
        stats["calls"] += 1

        result = await Runner.run(writer_agent, notes)
        report = result.final_output
        stats["calls"] += 1

        result = await Runner.run(reviewer_agent, report)
        feedback = result.final_output
        stats["calls"] += 1

        print(f"Iteration {i+1}: {len(report)} chars, {stats['calls']} total calls")
        notes = f"Previous report:\n{report}\n\nReviewer feedback:\n{feedback}"

    stats["elapsed"] = time.time() - stats["start"]
    stats["report_len"] = len(report)
    return stats, report

In [ ]:
uncontained_stats, uncontained_report = asyncio.run(
    run_uncontained("Compare WebSocket vs SSE for real-time AI streaming applications")
)
print(f"\nTotal calls: {uncontained_stats['calls']}")
print(f"Elapsed: {uncontained_stats['elapsed']:.1f}s")

All 6 iterations ran. The reviewer kept finding things to improve, so the loop never stopped early. In production, this means unbounded cost for every request.

## Part 2: With Containment

Now we wrap the same loop with `veronica-core`. Two enforcement mechanisms:

- **BudgetEnforcer**: tracks spend and halts when the dollar cap is hit
- **CircuitBreaker**: opens after N consecutive failures, skipping the broken agent

4. Set up budget and circuit breaker

In [ ]:
from veronica_core import BudgetEnforcer, CircuitBreaker, CircuitState

budget = BudgetEnforcer(limit_usd=0.50)
print(f"Budget: ${budget.limit_usd:.2f}, spent: ${budget.spent_usd:.2f}")

5. Wrap the runner with enforcement

In [ ]:
class ContainedRunner:
    def __init__(self, budget, failure_threshold=3):
        self.budget = budget
        self.failure_threshold = failure_threshold
        self.call_count = 0
        self.blocked_calls = []
        self.breakers = {}

    def _get_breaker(self, name):
        if name not in self.breakers:
            self.breakers[name] = CircuitBreaker(
                failure_threshold=self.failure_threshold,
            )
        return self.breakers[name]

    async def run(self, agent, input_text):
        name = agent.name
        cb = self._get_breaker(name)

        # Circuit breaker: skip agents that tripped
        if cb.state == CircuitState.OPEN:
            self.blocked_calls.append((name, "circuit_open"))
            return None

        # Budget: halt if we already blew the cap
        if self.budget.is_exceeded:
            self.blocked_calls.append((name, "budget_exceeded"))
            return None

        try:
            result = await Runner.run(agent, input_text)
            self.call_count += 1
            # spend() records cost after the call completes -- the call
            # that crosses the limit still runs, but the next one is blocked
            self.budget.spend(0.05)  # estimate per call
            cb.record_success()
            return result
        except Exception:
            cb.record_failure()
            self.blocked_calls.append((name, f"error (failures: {cb.failure_count})"))
            return None

6. Run the contained loop

In [ ]:
async def run_contained(topic, max_iterations=6):
    budget = BudgetEnforcer(limit_usd=0.50)
    runner = ContainedRunner(budget)
    stats = {"start": time.time()}
    notes = topic
    report = ""

    for i in range(max_iterations):
        result = await runner.run(research_agent, notes)
        if result is None:
            print(f"Iteration {i+1}: HALTED (budget or circuit breaker)")
            break
        notes = result.final_output

        result = await runner.run(writer_agent, notes)
        if result is None:
            print(f"Iteration {i+1}: HALTED after research")
            break
        report = result.final_output

        result = await runner.run(reviewer_agent, report)
        if result is None:
            print(f"Iteration {i+1}: HALTED after write")
            break
        feedback = result.final_output

        print(f"Iteration {i+1}: {len(report)} chars, {runner.call_count} calls, ${runner.budget.spent_usd:.2f} spent")
        notes = f"Previous report:\n{report}\n\nReviewer feedback:\n{feedback}"

    stats["calls"] = runner.call_count
    stats["blocked"] = len(runner.blocked_calls)
    stats["elapsed"] = time.time() - stats["start"]
    stats["spent_usd"] = runner.budget.spent_usd
    stats["report_len"] = len(report) if report else 0
    return stats, report, runner

In [ ]:
contained_stats, contained_report, runner = asyncio.run(
    run_contained("Compare WebSocket vs SSE for real-time AI streaming applications")
)
print(f"\nTotal calls: {contained_stats['calls']}")
print(f"Blocked: {contained_stats['blocked']}")
print(f"Spent: ${contained_stats['spent_usd']:.2f} / $0.50 budget")
print(f"Elapsed: {contained_stats['elapsed']:.1f}s")
if runner.blocked_calls:
    print(f"Block log: {runner.blocked_calls}")

The loop stopped after the budget was exhausted. The report from the last complete iteration is still usable -- we just did not burn more API calls polishing it.

## Part 3: Before/After Comparison

7. Compare the two runs

In [ ]:
header = f"{'Metric':<25} {'Without':<20} {'With':<20}"
print(header)
print("-" * 65)
rows = [
    ("LLM calls", uncontained_stats["calls"], contained_stats["calls"]),
    ("Blocked calls", 0, contained_stats["blocked"]),
    ("Elapsed (s)", f"{uncontained_stats['elapsed']:.1f}", f"{contained_stats['elapsed']:.1f}"),
    ("Report length (chars)", uncontained_stats["report_len"], contained_stats["report_len"]),
]
for metric, without, with_ in rows:
    print(f"{metric:<25} {str(without):<20} {str(with_):<20}")

## Summary

The uncontained loop ran all iterations and made ~18 LLM calls. The contained version stopped when the $0.50 budget was hit and still produced a usable report.

- **Budget enforcement** prevents runaway cost. Set a dollar cap per request, and the loop halts cleanly instead of silently overspending.
- **Circuit breaking** isolates failing agents. If one agent starts erroring, it gets removed from the loop instead of poisoning every iteration.
- **These are complementary to content guardrails.** OpenAI's built-in guardrails handle what the model *says*. Runtime containment handles what the system *does*.